# Notebook 05 — Authentication, Cost Tracking & LangSmith Monitoring

## Objectives
- Implement simple API key authentication
- Understand JWT-based auth for production FastAPI
- Track and estimate LLM API costs
- Set up LangSmith end-to-end tracing
- Use `day5.deployment` patterns for production-ready auth and monitoring

## 1. Simple API key authentication

In [ ]:
import os
import time

VALID_API_KEYS = {"sk-demo-key-123", "sk-prod-key-456"}

def check_auth(api_key: str) -> dict:
    if api_key in VALID_API_KEYS:
        return {"authenticated": True, "tier": "pro" if "prod" in api_key else "demo"}
    return {"authenticated": False, "error": "Invalid API key"}

print(check_auth("sk-demo-key-123"))
print(check_auth("sk-prod-key-456"))
print(check_auth("sk-invalid-key"))

## JWT-based auth for production APIs

API key auth is simple but doesn't carry user metadata or expiry. JWTs solve this:

- **Header**: algorithm + token type
- **Payload**: user ID, role, expiry (`exp`), issued at (`iat`)
- **Signature**: HMAC-SHA256 of header + payload, signed with SECRET_KEY

In FastAPI, you add a `Depends(verify_token)` to protected routes.
The `HTTPBearer` security scheme extracts the `Authorization: Bearer <token>` header automatically.

## 2. JWT pattern from deployment.py

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
from day5.deployment import show_auth_pattern

print(show_auth_pattern())

## 3. Cost tracking — why it matters

LLM API costs can spiral quickly in production:

| Model | Input (per 1M tokens) | Output (per 1M tokens) |
|-------|----------------------|------------------------|
| gpt-4o-mini | $0.15 | $0.60 |
| gpt-4o | $5.00 | $15.00 |
| claude-3-haiku | $0.25 | $1.25 |

A production app handling 10,000 queries/day with ~1000 tokens each costs:
- gpt-4o-mini: ~$7.50/day = ~$225/month
- gpt-4o: ~$200/day = ~$6,000/month

Cost tracking lets you set budgets, throttle expensive users, and choose models intelligently.

## 4. Manual cost calculation

In [ ]:
# OpenAI pricing (gpt-4o-mini)
PRICE_PER_1K_INPUT  = 0.000150  # $0.15 per 1M tokens
PRICE_PER_1K_OUTPUT = 0.000600  # $0.60 per 1M tokens

def estimate_cost(input_tokens, output_tokens, model="gpt-4o-mini"):
    cost = (input_tokens * PRICE_PER_1K_INPUT + output_tokens * PRICE_PER_1K_OUTPUT) / 1000
    return {"model": model, "input_tokens": input_tokens, "output_tokens": output_tokens, "cost_usd": round(cost, 6)}

print(estimate_cost(1000, 500))
print(estimate_cost(10000, 2000))
print()

# Daily projection
queries_per_day = 10_000
avg_input  = 800
avg_output = 300
daily = queries_per_day * (avg_input * PRICE_PER_1K_INPUT + avg_output * PRICE_PER_1K_OUTPUT) / 1000
print(f"Projected daily cost  ({queries_per_day:,} queries): ${daily:.2f}")
print(f"Projected monthly cost                           : ${daily * 30:.2f}")

## 5. LangSmith tracing

LangSmith automatically traces all LangChain/LangGraph calls when `LANGCHAIN_TRACING_V2=true` is set.

It records:
- Input/output of every node
- Token counts and latency
- Full conversation threads
- Errors and retries

For custom spans, use the `@traceable` decorator.

In [ ]:
from day5.deployment import show_langsmith_tracing

print(show_langsmith_tracing())

## 6. Using day5.deployment auth + checklist

In [ ]:
from day5.deployment import show_auth_pattern, get_deployment_checklist

print("Production Deployment Checklist:")
print("=" * 50)
for item in get_deployment_checklist():
    print(f"  {item}")

## 7. Putting it all together: monitored query handler

In [ ]:
import time
from day5.evaluation import RAGMonitor, CostTracker
from day5.multi_agent_orchestrator import build_orchestrator, run_orchestrator

monitor = RAGMonitor(faithfulness_threshold=0.5)
costs   = CostTracker(model="gpt-4o-mini")

docs = [
    "LangGraph builds stateful multi-agent workflows with conditional routing",
    "BM25 is a keyword ranking algorithm for information retrieval",
    "Hybrid search combines BM25 and dense semantic embeddings",
]

def retrieve(q):
    return [d for d in docs if any(w in d.lower() for w in q.lower().split())][:2]

graph = build_orchestrator(retrieve_fn=retrieve)

test_queries = [
    "What is BM25?",
    "How does hybrid search work?",
    "Explain LangGraph workflows",
]

for q in test_queries:
    t0 = time.time()
    result = run_orchestrator(graph, q)
    latency = (time.time() - t0) * 1000
    
    tokens = result.get("cost_tokens", 50)
    eval_result = monitor.record(q, result["synthesis"], result["retrieved_docs"], tokens=tokens, latency_ms=latency)
    cost = costs.record(tokens, tokens // 3, q)
    
    print(f"Q: {q[:35]:35s} faith={eval_result.faithfulness:.2f} cost=${cost:.6f}")

print()
metrics = monitor.get_metrics()
print(f"Avg faithfulness: {metrics.avg_faithfulness}")
print(f"Total cost      : ${costs.total_cost():.6f}")